In [1]:
# Library bawaan Python untuk operasi regex (pembersihan teks) dan operasi matematika (log untuk IDF)
import re
import math

# Pandas digunakan untuk menampilkan dan mengolah data dalam bentuk tabel (DataFrame)
import pandas as pd

# TfidfVectorizer dari scikit-learn: implementasi TF-IDF versi library, sebagai pembanding hitungan manual
from sklearn.feature_extraction.text import TfidfVectorizer

# Sastrawi: library bahasa Indonesia untuk menghapus stopword (kata umum yang tidak informatif, mis. 'yang', 'untuk')
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Membuat objek stopword remover bahasa Indonesia
stopword_factory = StopWordRemoverFactory()
stopword_remover = stopword_factory.create_stop_word_remover()

# Melebarkan tampilan kolom pandas agar tabel tidak terpotong saat ditampilkan
pd.set_option('display.width', 120)

In [2]:
# Kumpulan dokumen (corpus) yang akan diproses.
# Masing-masing string mewakili satu dokumen dengan topik berbeda.
documents = [
    "Sistem komputer modern menggunakan prosesor multicore untuk mempercepat kinerja.",
    "Jaringan komputer menghubungkan banyak perangkat melalui protokol internet.",
    "Kecerdasan buatan mempelajari pola data untuk menghasilkan prediksi yang cerdas.",
    "Sistem temu kembali informasi membantu pengguna menemukan dokumen yang relevan dengan cepat.",
]

# Menampilkan setiap dokumen beserta labelnya (D1, D2, D3, D4)
for i, doc in enumerate(documents, 1):
    print(f"D{i}: {doc}")

D1: Sistem komputer modern menggunakan prosesor multicore untuk mempercepat kinerja.
D2: Jaringan komputer menghubungkan banyak perangkat melalui protokol internet.
D3: Kecerdasan buatan mempelajari pola data untuk menghasilkan prediksi yang cerdas.
D4: Sistem temu kembali informasi membantu pengguna menemukan dokumen yang relevan dengan cepat.


In [3]:
def simple_preprocess(text):
    # 1. Case folding: ubah semua huruf menjadi huruf kecil
    text = text.lower()

    # 2. Hilangkan tanda baca dan angka, sisakan huruf dan spasi saja
    text = re.sub(r'[^a-z\s]', ' ', text)

    # 3. Rapikan spasi berlebih (misalnya akibat penghapusan tanda baca) menjadi satu spasi
    text = re.sub(r'\s+', ' ', text).strip()

    # 4. Hapus stopword bahasa Indonesia (kata seperti 'untuk', 'yang', 'dengan', dst.)
    text = stopword_remover.remove(text)

    # 5. Tokenisasi: pecah kalimat menjadi daftar kata (token) berdasarkan spasi
    tokens = text.split()
    return tokens

# Terapkan fungsi preprocessing ke seluruh dokumen -> hasilnya daftar token per dokumen
processed_docs = [simple_preprocess(doc) for doc in documents]

# Gabungkan kembali token menjadi string (dibutuhkan nanti oleh TfidfVectorizer)
processed_strings = [' '.join(tokens) for tokens in processed_docs]

# Tampilkan hasil token tiap dokumen setelah preprocessing
for i, tokens in enumerate(processed_docs, 1):
    print(f"D{i}:", tokens)

D1: ['sistem', 'komputer', 'modern', 'menggunakan', 'prosesor', 'multicore', 'mempercepat', 'kinerja']
D2: ['jaringan', 'komputer', 'menghubungkan', 'banyak', 'perangkat', 'melalui', 'protokol', 'internet']
D3: ['kecerdasan', 'buatan', 'mempelajari', 'pola', 'data', 'menghasilkan', 'prediksi', 'cerdas']
D4: ['sistem', 'temu', 'informasi', 'membantu', 'pengguna', 'menemukan', 'dokumen', 'relevan', 'cepat']


In [4]:
# Bangun kosakata (vocabulary): kumpulan semua term unik dari seluruh dokumen, diurutkan alfabetis
vocab = sorted(set(term for doc in processed_docs for term in doc))

# Label baris untuk tiap dokumen: D1, D2, D3, D4
doc_labels = [f"D{i+1}" for i in range(len(documents))]

# Buat tabel kosong (semua nol) dengan baris = dokumen, kolom = term pada vocabulary
bow = pd.DataFrame(0, index=doc_labels, columns=vocab)

# Isi tabel: untuk tiap dokumen, hitung berapa kali tiap term muncul (raw count)
for i, tokens in enumerate(processed_docs):
    for term in tokens:
        bow.loc[doc_labels[i], term] += 1

# Tampilkan tabel Bag-of-Words
bow

,banyak,buatan,cepat,cerdas,data,dokumen,informasi,internet,jaringan,kecerdasan,...,multicore,pengguna,perangkat,pola,prediksi,prosesor,protokol,relevan,sistem,temu
D1,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,1,0,0,1,0
D2,1,0,0,0,0,0,0,1,1,0,...,0,0,1,0,0,0,1,0,0,0
D3,0,1,0,1,1,0,0,0,0,1,...,0,0,0,1,1,0,0,0,0,0
D4,0,0,1,0,0,1,1,0,0,0,...,0,1,0,0,0,0,0,1,1,1


In [5]:
# N = jumlah total dokumen dalam corpus
N = len(documents)

# TF (Term Frequency) manual = sama persis dengan tabel Bag-of-Words (raw count)
tf_manual = bow.copy()

# DF (Document Frequency): untuk tiap term, hitung di berapa dokumen term itu muncul (nilai > 0)
df_manual = (bow > 0).sum(axis=0)

# IDF (Inverse Document Frequency): log(N / DF(t)) -> semakin jarang term muncul di banyak dokumen, semakin tinggi nilainya
idf_manual = df_manual.apply(lambda df_t: math.log(N / df_t))

# TF-IDF manual = TF dikalikan IDF untuk setiap kolom (term)
tfidf_manual = tf_manual.multiply(idf_manual, axis=1)

# Tampilkan hasil TF
print("Term Frequency (TF):")
display(tf_manual)

# Tampilkan hasil DF dan IDF berdampingan
print("\nDocument Frequency (DF) dan Inverse Document Frequency (IDF):")
df_idf_table = pd.DataFrame({"DF": df_manual, "IDF": idf_manual.round(4)})
display(df_idf_table)

# Tampilkan matriks TF-IDF hasil perhitungan manual (dibulatkan 4 desimal)
print("\nMatriks TF-IDF (Manual):")
display(tfidf_manual.round(4))

Term Frequency (TF):


,banyak,buatan,cepat,cerdas,data,dokumen,informasi,internet,jaringan,kecerdasan,...,multicore,pengguna,perangkat,pola,prediksi,prosesor,protokol,relevan,sistem,temu
D1,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,1,0,0,1,0
D2,1,0,0,0,0,0,0,1,1,0,...,0,0,1,0,0,0,1,0,0,0
D3,0,1,0,1,1,0,0,0,0,1,...,0,0,0,1,1,0,0,0,0,0
D4,0,0,1,0,0,1,1,0,0,0,...,0,1,0,0,0,0,0,1,1,1



Document Frequency (DF) dan Inverse Document Frequency (IDF):


,DF,IDF
banyak,1,1.3863
buatan,1,1.3863
cepat,1,1.3863
cerdas,1,1.3863
data,1,1.3863
dokumen,1,1.3863
informasi,1,1.3863
internet,1,1.3863
jaringan,1,1.3863
kecerdasan,1,1.3863



Matriks TF-IDF (Manual):


,banyak,buatan,cepat,cerdas,data,dokumen,informasi,internet,jaringan,kecerdasan,...,multicore,pengguna,perangkat,pola,prediksi,prosesor,protokol,relevan,sistem,temu
D1,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,1.3863,0.0000,0.0000,0.0000,0.0000,1.3863,0.0000,0.0000,0.6931,0.0000
D2,1.3863,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.3863,1.3863,0.0000,...,0.0000,0.0000,1.3863,0.0000,0.0000,0.0000,1.3863,0.0000,0.0000,0.0000
D3,0.0000,1.3863,0.0000,1.3863,1.3863,0.0000,0.0000,0.0000,0.0000,1.3863,...,0.0000,0.0000,0.0000,1.3863,1.3863,0.0000,0.0000,0.0000,0.0000,0.0000
D4,0.0000,0.0000,1.3863,0.0000,0.0000,1.3863,1.3863,0.0000,0.0000,0.0000,...,0.0000,1.3863,0.0000,0.0000,0.0000,0.0000,0.0000,1.3863,0.6931,1.3863


In [6]:
# Buat vectorizer dengan vocabulary yang SAMA seperti perhitungan manual, agar hasilnya bisa dibandingkan apple-to-apple
vectorizer = TfidfVectorizer(vocabulary=vocab, token_pattern=r"(?u)\b\w+\b")

# Fit + transform teks yang sudah dipreprocess menjadi matriks TF-IDF (sparse matrix)
tfidf_sklearn_matrix = vectorizer.fit_transform(processed_strings)

# Ubah sparse matrix menjadi DataFrame agar mudah dibaca, dengan baris = dokumen, kolom = term
tfidf_sklearn = pd.DataFrame(
    tfidf_sklearn_matrix.toarray(),
    index=doc_labels,
    columns=vectorizer.get_feature_names_out()
)

# Tampilkan matriks TF-IDF hasil scikit-learn
print("Matriks TF-IDF (scikit-learn, default settings):")
tfidf_sklearn.round(4)

Matriks TF-IDF (scikit-learn, default settings):


,banyak,buatan,cepat,cerdas,data,dokumen,informasi,internet,jaringan,kecerdasan,...,multicore,pengguna,perangkat,pola,prediksi,prosesor,protokol,relevan,sistem,temu
D1,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,...,0.3716,0.0000,0.0000,0.0000,0.0000,0.3716,0.0000,0.0000,0.2929,0.0000
D2,0.3622,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.3622,0.3622,0.0000,...,0.0000,0.0000,0.3622,0.0000,0.0000,0.0000,0.3622,0.0000,0.0000,0.0000
D3,0.0000,0.3536,0.0000,0.3536,0.3536,0.0000,0.0000,0.0000,0.0000,0.3536,...,0.0000,0.0000,0.0000,0.3536,0.3536,0.0000,0.0000,0.0000,0.0000,0.0000
D4,0.0000,0.0000,0.3406,0.0000,0.0000,0.3406,0.3406,0.0000,0.0000,0.0000,...,0.0000,0.3406,0.0000,0.0000,0.0000,0.0000,0.0000,0.3406,0.2685,0.3406


In [7]:
# Gabungkan kedua tabel (manual & scikit-learn) menjadi satu tabel bertingkat (multi-index)
# agar mudah dibandingkan baris per baris untuk tiap dokumen
comparison = pd.concat(
    {"Manual": tfidf_manual.round(4), "Scikit-learn": tfidf_sklearn.round(4)},
    axis=0
)
comparison

banyak  buatan   cepat  cerdas    data  dokumen  informasi  internet  jaringan  kecerdasan  ...  \
Manual       D1  0.0000  0.0000  0.0000  0.0000  0.0000   0.0000     0.0000    0.0000    0.0000      0.0000  ...   
             D2  1.3863  0.0000  0.0000  0.0000  0.0000   0.0000     0.0000    1.3863    1.3863      0.0000  ...   
             D3  0.0000  1.3863  0.0000  1.3863  1.3863   0.0000     0.0000    0.0000    0.0000      1.3863  ...   
             D4  0.0000  0.0000  1.3863  0.0000  0.0000   1.3863     1.3863    0.0000    0.0000      0.0000  ...   
Scikit-learn D1  0.0000  0.0000  0.0000  0.0000  0.0000   0.0000     0.0000    0.0000    0.0000      0.0000  ...   
             D2  0.3622  0.0000  0.0000  0.0000  0.0000   0.0000     0.0000    0.3622    0.3622      0.0000  ...   
             D3  0.0000  0.3536  0.0000  0.3536  0.3536   0.0000     0.0000    0.0000    0.0000      0.3536  ...   
             D4  0.0000  0.0000  0.3406  0.0000  0.0000   0.3406     0.3406    0.0000    0.0000      0.0000  ...   

                 multicore  pengguna  perangkat    pola  prediksi  prosesor  protokol  relevan  sistem    temu  
Manual       D1     1.3863    0.0000     0.0000  0.0000    0.0000    1.3863    0.0000   0.0000  0.6931  0.0000  
             D2     0.0000    0.0000     1.3863  0.0000    0.0000    0.0000    1.3863   0.0000  0.0000  0.0000  
             D3     0.0000    0.0000     0.0000  1.3863    1.3863    0.0000    0.0000   0.0000  0.0000  0.0000  
             D4     0.0000    1.3863     0.0000  0.0000    0.0000    0.0000    0.0000   1.3863  0.6931  1.3863  
Scikit-learn D1     0.3716    0.0000     0.0000  0.0000    0.0000    0.3716    0.0000   0.0000  0.2929  0.0000  
             D2     0.0000    0.0000     0.3622  0.0000    0.0000    0.0000    0.3622   0.0000  0.0000  0.0000  
             D3     0.0000    0.0000     0.0000  0.3536    0.3536    0.0000    0.0000   0.0000  0.0000  0.0000  
             D4     0.0000    0.3406     0.0000  0.0000    0.0000    0.0000    0.0000   0.3406  0.2685  0.3406  

[8 rows x 31 columns]

In [8]:
# Untuk tiap dokumen, cari term dengan bobot TF-IDF (manual) tertinggi
print("Berdasarkan perhitungan manual:")
for doc_label in doc_labels:
    row = tfidf_manual.loc[doc_label]
    top_term = row.idxmax()          # nama term dengan bobot tertinggi
    print(f"  {doc_label}: '{top_term}' (bobot = {row.max():.4f})")

# Lakukan hal yang sama untuk hasil scikit-learn, sebagai pembanding
print("\nBerdasarkan scikit-learn:")
for doc_label in doc_labels:
    row = tfidf_sklearn.loc[doc_label]
    top_term = row.idxmax()
    print(f"  {doc_label}: '{top_term}' (bobot = {row.max():.4f})")

Berdasarkan perhitungan manual:
  D1: 'kinerja' (bobot = 1.3863)
  D2: 'banyak' (bobot = 1.3863)
  D3: 'buatan' (bobot = 1.3863)
  D4: 'cepat' (bobot = 1.3863)

Berdasarkan scikit-learn:
  D1: 'kinerja' (bobot = 0.3716)
  D2: 'banyak' (bobot = 0.3622)
  D3: 'buatan' (bobot = 0.3536)
  D4: 'cepat' (bobot = 0.3406)


## Analisis Singkat

Term dengan bobot TF-IDF tertinggi pada masing-masing dokumen (lihat hasil pada bagian 8) umumnya
adalah term yang bersifat unik atau spesifik terhadap dokumen tersebut — kata itu hanya muncul di satu
dokumen (nilai *document frequency* rendah, sehingga IDF-nya tinggi) sekaligus cukup sering muncul pada
dokumen itu sendiri (TF ≥ 1). Term-term umum yang muncul di banyak dokumen, seperti "komputer" atau
"sistem" pada dataset ini, justru mendapat bobot IDF yang lebih rendah karena kurang memiliki daya
pembeda antar dokumen. Term dengan bobot tinggi inilah yang paling merepresentasikan "topik unik" dari
sebuah dokumen, sehingga sangat berguna sebagai fitur pembeda saat sistem menghitung kemiripan
(*cosine similarity*) antar dokumen atau antara dokumen dengan query pengguna dalam sistem temu
kembali informasi.